# Modeling

In [74]:
import pandas as pd
from sklearn.decomposition import PCA
import numpy as np

from sklearn.linear_model import Ridge, Lasso, ElasticNet, BayesianRidge, HuberRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor,
    ExtraTreesRegressor
)

#import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn import model_selection
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV


from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error


In [75]:
df_train = pd.read_csv('../data/data_model/train_model.csv', sep=',')


In [76]:
X_train = df_train.drop(columns=['SALEPRICE'])
y_train = df_train['SALEPRICE']

In [77]:
df_train.head()

,NEIGHBORHOOD,OVERALLQUAL,EXTERQUAL,BSMTQUAL,HEATINGQC,GRLIVAREA,KITCHENQUAL,TOTRMSABVGRD,GARAGEYRBLT,GARAGEFINISH,...,EXTERIOR1ST_OTHER,EXTERIOR1ST_PLYWOOD,EXTERIOR1ST_VINYLSD,EXTERIOR1ST_WD SDNG,GARAGETYPE_DETCHD,GARAGETYPE_OTHER,SALETYPE_CON,SALETYPE_NEW,SALETYPE_WD,SALEPRICE
0,0.483681,0.550585,0.930835,0.491996,0.814688,0.531594,0.636941,0.969273,0.916845,0.123461,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,12.247699
1,1.142755,-0.233625,-0.827054,0.491996,0.814688,-0.472592,-0.935418,-0.338533,-0.282761,0.123461,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,12.109016
2,0.483681,0.550585,0.930835,0.491996,0.814688,0.675348,0.636941,-0.338533,0.827986,0.123461,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,12.317171
3,1.801830,1.334796,0.930835,0.491996,0.814688,1.361584,0.636941,1.623175,0.783556,0.123461,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,12.429220
4,0.977986,1.334796,0.930835,1.996684,0.814688,0.500517,0.636941,0.315370,0.961275,0.123461,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,12.634606


In [78]:
pca_analysis = PCA(random_state=0) 
pca_analysis.fit(X_train)

individual_variance = pca_analysis.explained_variance_ratio_
accumulated_variance = individual_variance.cumsum()

pca_table = pd.DataFrame({
    'COMPONENT': range(1, len(accumulated_variance) + 1),
    'INDIVIDUAL_VARIANCE': (individual_variance).round(4),
    'ACCUMULATED_VARIANCE': (accumulated_variance).round(4),
    'DELTA': (pd.Series(accumulated_variance).diff().fillna(accumulated_variance[0]) * 100).round(4)
})

first_above_95 = np.argmax(np.array(pca_table['ACCUMULATED_VARIANCE']) >= 0.95)
first_above_95

np.int64(16)

In [79]:
pca = PCA(n_components=16, random_state=0)
X_train_pca = pca.fit_transform(X_train) 

In [80]:
models = {
    'RIDGE':                   Ridge(),
    'LASSO':                   Lasso(),
    'ELASTICNET':              ElasticNet(),
    'BAYESIAN RIDGE':          BayesianRidge(),
    'HUBER':                   HuberRegressor(max_iter=1000),
    'KNN REGRESSOR':           KNeighborsRegressor(),
    'SVR':                     SVR(),
    'RANDOM FOREST':           RandomForestRegressor(random_state=0),
    'EXTRA TREES':             ExtraTreesRegressor(random_state=0),
    'GRADIENT BOOSTING':       GradientBoostingRegressor(random_state=0),
    #'XGBOOST':                 xgb.XGBRegressor(random_state=0)
}

kfold = model_selection.KFold(
    n_splits=10,
    shuffle=True,
    random_state=0
)

results = []

for name, model in models.items():

    pipeline = Pipeline([
        ('model', model)
    ])

    scores = model_selection.cross_validate(
        pipeline,
        X_train,
        y_train,
        scoring=['r2', 'neg_mean_absolute_error', 'neg_mean_squared_error'],
        cv=kfold,
        n_jobs=-1
    )

    r2   = round(scores['test_r2'].mean(), 4)
    mae  = round(-scores['test_neg_mean_absolute_error'].mean(), 4)
    rmse = round(np.sqrt(-scores['test_neg_mean_squared_error']).mean(), 4)

    result = {
        'name': name,
        'r2':   r2,
        'mae':  mae,
        'rmse': rmse
    }
    results.append(result)

In [81]:
df_exploracao = pd.DataFrame(results)
df_exploracao.columns = df_exploracao.columns.str.upper()
df_exploracao = df_exploracao.round(4)

df_exploracao.sort_values(by=['R2'], ascending=False)

,NAME,R2,MAE,RMSE
9,GRADIENT BOOSTING,0.8782,0.0868,0.1196
8,EXTRA TREES,0.8779,0.0848,0.1200
7,RANDOM FOREST,0.8725,0.0878,0.1223
0,RIDGE,0.8702,0.0928,0.1236
3,BAYESIAN RIDGE,0.8699,0.0927,0.1237
6,SVR,0.8694,0.0888,0.1239
4,HUBER,0.8688,0.0918,0.1242
5,KNN REGRESSOR,0.8384,0.0988,0.1377
1,LASSO,-0.0123,0.2744,0.3455
2,ELASTICNET,-0.0123,0.2744,0.3455


In [82]:
df_exploracao_analysis = df_exploracao.copy()
df_exploracao_analysis[['MAE', 'RMSE']] = df_exploracao_analysis[['MAE', 'RMSE']].rank(0, numeric_only=True, ascending=True)
df_exploracao_analysis['R2'] = df_exploracao_analysis['R2'].rank(0, numeric_only=True, ascending=False)

df_exploracao_analysis['POINTS'] = df_exploracao_analysis['R2'] +  df_exploracao_analysis['MAE'] + df_exploracao_analysis['RMSE'] 
df_exploracao_analysis = df_exploracao_analysis.sort_values(by=['POINTS'], ascending=True)
df_exploracao_analysis = df_exploracao_analysis.reset_index(drop=True)

df_exploracao_analysis.head(3)

,NAME,R2,MAE,RMSE,POINTS
0,GRADIENT BOOSTING,1.0,2.0,1.0,4.0
1,EXTRA TREES,2.0,1.0,2.0,5.0
2,RANDOM FOREST,3.0,3.0,3.0,9.0


## GRADIENT BOOSTING

In [83]:
gb_model = GradientBoostingRegressor(random_state=0)

param_distributions = {
    "n_estimators": [100, 300, 500],
    "learning_rate": [0.05, 0.1, 0.2],
    "max_depth": [5, 3, 7], # experimentacao
    "min_samples_split": [22, 56, 110], #dobro do min_sample_leaf
    "min_samples_leaf": [11, 28, 55], # Geralmente, na faixa de 1-5%
    "max_features": [0.25, 0.75, 0.8, 1.0], # um valor classico é \sqrt (no caso, 4 = 0.25)
    "subsample": [0.75, 0.85, 0.95] # depende do tamanho do modelo, cria generalização
}

random_search = RandomizedSearchCV(
    estimator = gb_model,
    param_distributions = param_distributions,
    n_jobs = -1,
    random_state = 0,
    scoring='d2_absolute_error_score'
)

random_search.fit(X_train_pca, y_train)

model_best_gb = random_search.best_estimator_
model_best_gb_score = random_search.best_score_

print(model_best_gb_score)
model_best_gb

0.6721008590547142


,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.05
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",500
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",0.85
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",22
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",28
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",5
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",0
,"max_features max_features: {'sqrt', 'log2'}, int or float, default=NoneThe number of features to consider when looking for the best split:- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0]` and the features considered at each split will be `max(1, int(max_features * n_features_in_))`.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Choosing `max_features < n_features` leads to a reduction of varianceand an increase in bias.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction in

In [84]:
param_grid_refined = {
    "n_estimators": [500, 600],
    "learning_rate": [0.04, 0.05],
    "max_depth": [3, 4],
    "min_samples_split": [22, 30],
    "min_samples_leaf": [28, 36],
    "max_features": [0.8, 1.0],
    "subsample": [0.85, 0.90]
}

grid_search = GridSearchCV(
    estimator=gb_model,
    param_grid=param_grid_refined,
    n_jobs=-1,
    scoring='d2_absolute_error_score'
)

grid_search.fit(X_train_pca, y_train)

model_best_gb = grid_search.best_estimator_
model_best_gb_score = grid_search.best_score_

print(model_best_gb_score)
model_best_gb

0.6756394758838995


,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.05
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",500
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",0.85
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",22
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",28
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",0
,"max_features max_features: {'sqrt', 'log2'}, int or float, default=NoneThe number of features to consider when looking for the best split:- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0]` and the features considered at each split will be `max(1, int(max_features * n_features_in_))`.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Choosing `max_features < n_features` leads to a reduction of varianceand an increase in bias.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.8
,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to

In [85]:
y_pred = model_best_gb.predict(X_train_pca)

In [86]:
print(round(mean_absolute_error(y_true=y_train, y_pred=y_pred),4))
print(round(r2_score(y_true=y_train, y_pred=y_pred),4))
print(round(root_mean_squared_error(y_true=y_train, y_pred=y_pred), 4))

0.0505
0.9596
0.0694
